# 11. 동시성 / 비동기 (threading, multiprocessing, asyncio)

GIL이 뭔지, I/O bound와 CPU bound 작업에 뭘 써야 하는지, 그리고 추론 서버 코드에서 자주 보는 `async`/`await` 기초까지.

## GIL (Global Interpreter Lock) 한 줄 요약

CPython은 한 번에 **파이썬 바이트코드 한 스레드만** 실행하도록 잠그는 GIL이 있습니다.

- **I/O bound** (네트워크 요청, 파일 읽기, DB 대기 등 '기다리는' 작업): 기다리는 동안 GIL이 풀리므로 `threading`이 효과적
- **CPU bound** (숫자 계산, 이미지 처리 등 '계산하는' 작업): GIL 때문에 스레드로는 병렬화 안 됨 → `multiprocessing`으로 프로세스를 여러 개 띄워야 함

## threading: I/O bound 작업에 적합

In [1]:
import threading
import time

def fetch_sensor(sensor_id, delay):
    time.sleep(delay)  # 네트워크 대기를 흉내 (이 동안 GIL이 풀림)
    print(f"sensor {sensor_id} 응답 완료 ({delay}s)")

start = time.perf_counter()

threads = [threading.Thread(target=fetch_sensor, args=(i, 0.3)) for i in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()  # 모든 스레드가 끝날 때까지 대기

print(f"총 소요 시간: {time.perf_counter() - start:.2f}s (순차 실행이면 1.2s여야 함)")


sensor 0 응답 완료 (0.3s)
sensor 2 응답 완료 (0.3s)
sensor 3 응답 완료 (0.3s)
sensor 1 응답 완료 (0.3s)
총 소요 시간: 0.30s (순차 실행이면 1.2s여야 함)


## multiprocessing: CPU bound 작업에 적합

각 프로세스가 독립된 파이썬 인터프리터(= 독립된 GIL)를 가지므로 진짜 병렬 계산이 가능합니다. 단, 프로세스 생성 비용이 있고 데이터를 주고받으려면 직렬화(pickle)가 필요합니다.

In [2]:
from concurrent.futures import ProcessPoolExecutor
import time

def cpu_heavy(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(cpu_heavy, [5_000_000] * 4))
print(results)
print(f"병렬 실행 소요 시간: {time.perf_counter() - start:.2f}s")


[41666654166667500000, 41666654166667500000, 41666654166667500000, 41666654166667500000]
병렬 실행 소요 시간: 0.44s


## asyncio 기초: async def / await

스레드를 여러 개 띄우는 대신, **하나의 스레드 안에서** '기다리는 동안 다른 작업으로 전환'하는 방식입니다. 웹 서버/추론 API에서 많이 봅니다 (FastAPI가 내부적으로 이 모델을 씀).

In [3]:
import asyncio

async def fetch_async(sensor_id, delay):
    await asyncio.sleep(delay)  # 'await'는 여기서 다른 코루틴에게 제어권을 넘김
    print(f"sensor {sensor_id} 응답 완료 ({delay}s)")
    return sensor_id

async def main():
    start = asyncio.get_event_loop().time()
    # gather: 여러 코루틴을 동시에(concurrently) 실행하고 모두 끝나길 기다림
    results = await asyncio.gather(*[fetch_async(i, 0.3) for i in range(4)])
    print("결과:", results)
    print(f"총 소요 시간: {asyncio.get_event_loop().time() - start:.2f}s")

await main()  # 주피터 노트북은 이미 이벤트 루프가 돌고 있어서 top-level await 가능
              # 일반 .py 스크립트에서는 asyncio.run(main())을 사용


sensor 0 응답 완료 (0.3s)
sensor 1 응답 완료 (0.3s)
sensor 2 응답 완료 (0.3s)
sensor 3 응답 완료 (0.3s)
결과: [0, 1, 2, 3]
총 소요 시간: 0.30s


## 언제 뭘 쓸지 정리

| 상황 | 선택 |
|---|---|
| 네트워크 요청 여러 개 (requests 등 동기 라이브러리) | `threading` 또는 `ProcessPoolExecutor`의 I/O 버전인 `ThreadPoolExecutor` |
| 숫자 계산/이미지 전처리를 병렬로 | `multiprocessing` / `ProcessPoolExecutor` |
| 수천 개의 동시 연결을 다루는 서버 (추론 API 등) | `asyncio` 기반 프레임워크 (FastAPI, aiohttp) |
| 간단한 스크립트, 병목이 없음 | 그냥 순차 실행 — 동시성은 복잡도를 늘리는 비용이 있음을 기억 |

## 헷갈렸던 점 / 더 찾아볼 것

- `asyncio`는 '병렬'이 아니라 '동시성' — 싱글 스레드에서 전환하며 도는 것이라 CPU bound 작업에는 도움 안 됨
- 실제 추론 서빙에서는 모델 호출 자체가 GPU에서 블로킹되는 경우가 많아 `asyncio` + `ThreadPoolExecutor`를 같이 쓰는 패턴도 흔함